# 공모펀드 grain·capability·품질 계약 재현

## TL;DR

- 원천 grain: `(itm_no, prfd_attr_cd)`
- 논리 상품 grain: `itm_no`
- 정상 논리 상품: 11,138개
- 구조 손상: Excel source row 84,563 한 건
- 공모펀드 기본 검색 범위: `prvo_pbff_desc = 공모`

이 노트북은 운영 감사기와 같은 코드를 호출하며 원천 XLSX를 수정하지 않는다

## Context & Methods

전제:

- `PRODUCT_DATA_DIR` 환경 변수는 공식 금융상품 XLSX 8개가 있는 디렉터리
- 파일 스냅샷일은 2026-07-11
- 결측·빈 문자열·문자열 `NULL`은 서로 집계하되 모두 조건 충족값으로 사용하지 않음
- raw-row coverage가 아니라 `itm_no` product-grain coverage를 사용

현재 `gaeng3-dev` 환경에는 Jupyter 실행 의존성이 포함되지 않아 저장소 검증에서는
동일 분석을 `finance-data-audit --dataset fund`로 실행한다. 노트북 실행 환경을
추가한 뒤 마지막 셀의 명령으로 위에서 아래까지 재실행 가능

In [ ]:
import datetime as dt
import json
import os
from importlib import resources
from pathlib import Path

from finance_agent_core.audit.pipeline import audit_dataset
from finance_agent_core.audit.registry import DATASET_BY_NAME, resolve_inputs
from finance_agent_core.audit.verification import verify_expectations

raw_data_dir = os.environ.get("PRODUCT_DATA_DIR")
if not raw_data_dir:
    raise RuntimeError(
        "PRODUCT_DATA_DIR must point to the official raw workbook directory"
    )

spec = DATASET_BY_NAME["fund"]
data_path, schema_path = resolve_inputs(Path(raw_data_dir), spec)
audit = audit_dataset(spec, data_path, schema_path, dt.date(2026, 7, 11))
grain = audit["domain"]["product_grain"]

{
    "source": audit["inputs"]["data"]["name"],
    "raw_rows": audit["structure"]["data_rows"],
    "raw_columns": audit["structure"]["columns"],
    "invalid_source_rows": audit["quality"]["invalid_key_rows"],
    "logical_products": grain["valid_logical_products"],
    "rows_per_product": grain["rows_per_product"],
}

## Data & Results

아래 셀은 과제 기본 범위, 주요 결측·센티널과 장기 수익률 이상 범위를
product grain에서 확인

In [ ]:
fields = grain["fields"]
{
    "scope_counts": grain["scope_counts"],
    "aum": fields["fd_nast_suma"],
    "risk": fields["zrin_fd_ivst_risk_gcd"],
    "currency_hedge": fields["exchdg_yn"],
    "fund_type": fields["or_attr_desc"],
    "return_outliers": grain["return_outliers"],
}

## Quality regression

패키지의 공모펀드 expectation만 골라 현재 분석 결과와 비교

In [ ]:
expectation_resource = resources.files("finance_agent_core.audit").joinpath(
    "expectations.json"
)
all_expectations = json.loads(expectation_resource.read_text(encoding="utf-8"))
fund_expectations = {
    **all_expectations,
    "checks": [
        check
        for check in all_expectations["checks"]
        if check["path"].startswith("datasets.fund.")
    ],
}
verification = verify_expectations({"datasets": {"fund": audit}}, fund_expectations)
assert verification["passed"], verification
{
    "passed": verification["passed"],
    "passed_count": verification["passed_count"],
    "total_count": verification["total_count"],
}

## Takeaways

- `fund_product`, `fund_attribute`, `fund_quarantine` 세 테이블로 정규화
- 공모 11,115개만 공식 공모펀드 기본 검색 후보
- AUM 0과 운용속성 코드 `06`은 `UNKNOWN`
- 1주·1·3·6개월 수익률만 검색·정렬·집계 가능
- 장기 수익률은 공식 산식과 이상치 규칙 확인 전 표시 전용
- 정규화 DB·oracle·verifier 구현 전에는 fund QueryPlan 실행을 비활성화

## 향후 노트북 실행 명령

Jupyter를 별도 개발 의존성으로 승인한 경우:

```bash
PRODUCT_DATA_DIR="/path/to/1.금융상품" \
  conda run -n gaeng3-dev jupyter nbconvert \
  --execute --to notebook --inplace \
  notebooks/public-fund-contract-audit.ipynb
```

현재 필수 재현 경로:

```bash
conda run -n gaeng3-dev finance-data-audit \
  --data-dir "/path/to/1.금융상품" \
  --output-dir artifacts/data-audit \
  --dataset fund
```